# BITS Pilani - M.Tech (AIML)
## Machine Learning Assignment - 2
**Comparative Study of 5 Classification Models on UCI/Kaggle Heart Disease Dataset**

This notebook covers:
1. Dataset loading and exploratory data analysis
2. Data preprocessing & scaling
3. Model training: Logistic Regression, Decision Tree, KNN, Naive Bayes, SVM
4. Calculation of 6 evaluation metrics: Accuracy, AUC, Precision, Recall, F1, MCC
5. Model comparison and saving artifacts

In [1]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, confusion_matrix, classification_report, roc_curve
)

print('All dependencies imported successfully!')

In [2]:
# Load original dataset
df = pd.read_csv('../dataset/original_dataset.csv')
print(f'Shape of original dataset: {df.shape}')
df.head()

In [3]:
# Split features and target
features = [c for c in df.columns if c != 'target']
X = df[features]
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train shape: {X_train.shape}, Test shape: {X_test.shape}')

In [4]:
# Fit Scaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    'Logistic Regression': (LogisticRegression(max_iter=1000, random_state=42), True),
    'Decision Tree': (DecisionTreeClassifier(max_depth=6, random_state=42), False),
    'KNN': (KNeighborsClassifier(n_neighbors=7, weights='distance'), True),
    'Naive Bayes': (GaussianNB(), False),
    'Support Vector Machine': (SVC(kernel='rbf', C=1.5, probability=True, random_state=42), True)
}

results = []
for name, (model, use_scaled) in models.items():
    X_tr = X_train_scaled if use_scaled else X_train
    X_te = X_test_scaled if use_scaled else X_test
    model.fit(X_tr, y_train)
    pred = model.predict(X_te)
    proba = model.predict_proba(X_te)[:, 1]
    results.append({
        'ML Model Name': name,
        'Accuracy': round(accuracy_score(y_test, pred), 4),
        'AUC': round(roc_auc_score(y_test, proba), 4),
        'Precision': round(precision_score(y_test, pred, average='binary'), 4),
        'Recall': round(recall_score(y_test, pred, average='binary'), 4),
        'F1': round(f1_score(y_test, pred, average='binary'), 4),
        'MCC': round(matthews_corrcoef(y_test, pred), 4)
    })

summary_df = pd.DataFrame(results)
summary_df